# D2 — Week 7: Retrieval Stack & Graph Build

This notebook implements the D2 requirements using the completed D1 retrieval work as the base: ingestion, MongoDB, Qdrant, Neo4j, hybrid `/search`, dataflow diagram, metrics table, and top-k examples with citations.

**Before running:** put your PDF corpus in `data/pdfs/` and your D1 questions file in `data/question.xlsx`, or change the paths in the configuration cell.

In [ ]:
# Install packages in Colab / notebook environment
!pip -q install fastapi uvicorn pymongo qdrant-client neo4j sentence-transformers rank-bm25 pypdf pandas numpy scikit-learn tqdm python-dotenv openpyxl

## 1. Configuration

The included dataset uses `PDF_DIR=./data/pdfs` and `QUESTIONS_PATH=./data/question.xlsx`. For Colab, upload/extract this package first or change the paths to your Google Drive folder.

In [ ]:
import os, re, time, json, pickle, hashlib
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

# Change these paths if you are using Google Drive
PDF_DIR = os.getenv('PDF_DIR', './data/pdfs')
QUESTIONS_PATH = os.getenv('QUESTIONS_PATH', './data/question.xlsx')

MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017')
MONGO_DB = os.getenv('MONGO_DB', 'csai415_d2')
QDRANT_URL = os.getenv('QDRANT_URL', 'http://localhost:6333')
QDRANT_COLLECTION = os.getenv('QDRANT_COLLECTION', 'paper_chunks')
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password123')

EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL', 'sentence-transformers/all-MiniLM-L6-v2')
CHUNK_SIZE = 900
CHUNK_OVERLAP = 180
HYBRID_ALPHA = 0.55

Path('reports').mkdir(exist_ok=True)
print('PDF_DIR:', PDF_DIR)
print('QUESTIONS_PATH:', QUESTIONS_PATH)

## 2. PDF → text → chunks with provenance

In [ ]:
def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()

def stable_id(value: str) -> str:
    return hashlib.sha1(value.encode('utf-8')).hexdigest()[:16]

def read_pdf_pages(pdf_path: str) -> List[Dict]:
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        try:
            text = clean_text(page.extract_text())
        except Exception:
            text = ""
        if text:
            pages.append({'page': i, 'text': text})
    return pages

def guess_title(pages: List[Dict], filename: str) -> str:
    if not pages:
        return Path(filename).stem
    first = pages[0]['text']
    m = re.search(r"\babstract\b", first, flags=re.I)
    candidate = first[:m.start()].strip() if m else first[:220]
    candidate = re.sub(r"[^A-Za-z0-9 ,:;\-()]+", " ", candidate)
    return clean_text(candidate)[:180] or Path(filename).stem

def guess_authors(pages: List[Dict]) -> List[str]:
    if not pages:
        return ['Unknown Author']
    first = pages[0]['text'][:1200]
    m = re.search(r"\babstract\b", first, flags=re.I)
    before_abs = first[:m.start()] if m else first[:500]
    before_abs = re.sub(r"\S+@\S+", " ", before_abs)
    chunks = [clean_text(x) for x in re.split(r",|;| and ", before_abs)]
    authors = []
    for x in chunks[1:6]:
        if 2 <= len(x.split()) <= 5 and not re.search(r"university|department|abstract|school", x, re.I):
            authors.append(x[:80])
    return authors or ['Unknown Author']

def infer_topics(texts: List[str], filenames: List[str], max_topics: int = 3) -> Dict[str, List[str]]:
    vectorizer = TfidfVectorizer(stop_words='english', max_features=1500, ngram_range=(1, 2))
    X = vectorizer.fit_transform(texts)
    terms = np.array(vectorizer.get_feature_names_out())
    out = {}
    for i, name in enumerate(filenames):
        row = X[i].toarray().ravel()
        top = terms[row.argsort()[-max_topics:][::-1]].tolist() if row.max() > 0 else ['general ai']
        out[name] = [t.replace('_', ' ') for t in top]
    return out

def chunk_pages(pages: List[Dict], chunk_size=900, overlap=180) -> List[Dict]:
    chunks = []
    step = max(1, chunk_size - overlap)
    for p in pages:
        text = p['text']
        for start in range(0, len(text), step):
            chunk_text = text[start:start + chunk_size]
            if len(chunk_text.strip()) < 80:
                continue
            chunks.append({'text': chunk_text, 'page_start': p['page'], 'page_end': p['page']})
    return chunks

def build_corpus(pdf_dir: str):
    pdf_paths = sorted(Path(pdf_dir).glob('*.pdf'))
    if not pdf_paths:
        raise FileNotFoundError(f'No PDFs found in {pdf_dir}')
    raw_pages, paper_texts, paper_files = {}, [], []
    for path in tqdm(pdf_paths, desc='Reading PDFs'):
        pages = read_pdf_pages(str(path))
        raw_pages[path.name] = pages
        paper_texts.append(' '.join(p['text'] for p in pages[:3])[:5000])
        paper_files.append(path.name)
    topics_by_file = infer_topics(paper_texts, paper_files)
    documents, chunks = [], []
    for path in pdf_paths:
        pages = raw_pages[path.name]
        paper_id = stable_id(path.name)
        doc = {
            'paper_id': paper_id,
            'title': guess_title(pages, path.name),
            'authors': guess_authors(pages),
            'venue': 'Unknown Venue',
            'year': None,
            'doi': None,
            'pdf_path': str(path),
            'filename': path.name,
            'topics': topics_by_file.get(path.name, ['general ai']),
            'num_pages': len(pages),
            'provenance': {'source': 'local_pdf', 'path': str(path)},
        }
        documents.append(doc)
        for j, ch in enumerate(chunk_pages(pages, CHUNK_SIZE, CHUNK_OVERLAP)):
            chunk_id = f'{paper_id}_{j:04d}'
            chunks.append({
                'chunk_id': chunk_id,
                'paper_id': paper_id,
                'title': doc['title'],
                'filename': path.name,
                'text': ch['text'],
                'page_start': ch['page_start'],
                'page_end': ch['page_end'],
                'topics': doc['topics'],
                'authors': doc['authors'],
                'citation': f"{path.name}, p. {ch['page_start']}",
                'provenance': {'source': 'local_pdf', 'path': str(path), 'page': ch['page_start']},
            })
    return documents, chunks

documents, chunks = build_corpus(PDF_DIR)
print('Documents:', len(documents))
print('Chunks:', len(chunks))
pd.DataFrame(chunks).head()

## 3. Store metadata in MongoDB

In [ ]:
from pymongo import MongoClient, ASCENDING

mongo = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
mongo.admin.command('ping')
db = mongo[MONGO_DB]
db.documents.create_index([('paper_id', ASCENDING)], unique=True)
db.chunks.create_index([('chunk_id', ASCENDING)], unique=True)
db.chunks.create_index([('paper_id', ASCENDING)])

db.documents.delete_many({})
db.chunks.delete_many({})
db.documents.insert_many(documents)
db.chunks.insert_many(chunks)

print('Mongo documents:', db.documents.count_documents({}))
print('Mongo chunks:', db.chunks.count_documents({}))

## 4. Create embeddings and store in Qdrant

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

model = SentenceTransformer(EMBEDDING_MODEL)
probe = model.encode(['dimension check'], normalize_embeddings=True)
vector_size = int(probe.shape[1])

qdrant = QdrantClient(url=QDRANT_URL, timeout=30)
collections = [c.name for c in qdrant.get_collections().collections]
if QDRANT_COLLECTION in collections:
    qdrant.delete_collection(QDRANT_COLLECTION)
qdrant.create_collection(
    collection_name=QDRANT_COLLECTION,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)

batch_size = 64
points = []
for start in tqdm(range(0, len(chunks), batch_size), desc='Embedding chunks'):
    batch = chunks[start:start + batch_size]
    vectors = model.encode([c['text'] for c in batch], normalize_embeddings=True, show_progress_bar=False)
    for local_i, (chunk, vector) in enumerate(zip(batch, vectors)):
        payload = {k: v for k, v in chunk.items() if k != 'text'}
        payload['text'] = chunk['text'][:1500]
        points.append(PointStruct(id=start + local_i, vector=vector.tolist(), payload=payload))
    if len(points) >= 256:
        qdrant.upsert(collection_name=QDRANT_COLLECTION, points=points)
        points = []
if points:
    qdrant.upsert(collection_name=QDRANT_COLLECTION, points=points)

print('Qdrant collection:', QDRANT_COLLECTION)
print('Vector size:', vector_size)
print('Stored points:', len(chunks))

## 5. Build BM25 + dense hybrid `/search` logic

In [ ]:
tokenized = [c['text'].lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized)
with open('reports/bm25_index.pkl', 'wb') as f:
    pickle.dump({'bm25': bm25, 'chunks': chunks}, f)
print('BM25 index saved to reports/bm25_index.pkl')

def normalize_scores(score_dict):
    if not score_dict:
        return {}
    vals = np.array(list(score_dict.values()), dtype=float)
    lo, hi = float(vals.min()), float(vals.max())
    if hi <= lo:
        return {k: 1.0 for k in score_dict}
    return {k: (v - lo) / (hi - lo) for k, v in score_dict.items()}

def hybrid_search(query: str, top_k: int = 5, alpha: float = HYBRID_ALPHA):
    started = time.time()
    raw_bm25 = np.array(bm25.get_scores(query.lower().split()), dtype=float)
    bm25_idx = raw_bm25.argsort()[-max(50, top_k * 10):][::-1]
    bm25_scores = {chunks[i]['chunk_id']: float(raw_bm25[i]) for i in bm25_idx}
    bm25_scores = normalize_scores(bm25_scores)

    qvec = model.encode([query], normalize_embeddings=True)[0].tolist()
    dense_hits = qdrant.search(collection_name=QDRANT_COLLECTION, query_vector=qvec, limit=max(50, top_k * 10))
    dense_scores = {h.payload['chunk_id']: float(h.score) for h in dense_hits}
    dense_scores = normalize_scores(dense_scores)

    chunk_by_id = {c['chunk_id']: c for c in chunks}
    fused = []
    for cid in set(bm25_scores) | set(dense_scores):
        score = alpha * dense_scores.get(cid, 0.0) + (1 - alpha) * bm25_scores.get(cid, 0.0)
        ch = chunk_by_id[cid]
        fused.append({
            'score': float(score),
            'chunk_id': cid,
            'filename': ch['filename'],
            'title': ch['title'],
            'page_start': ch['page_start'],
            'page_end': ch['page_end'],
            'citation': ch['citation'],
            'text': ch['text'][:500],
            'bm25_score': float(bm25_scores.get(cid, 0.0)),
            'dense_score': float(dense_scores.get(cid, 0.0)),
        })
    fused.sort(key=lambda x: x['score'], reverse=True)
    return {'query': query, 'latency_ms': round((time.time() - started) * 1000, 2), 'results': fused[:top_k]}

example = hybrid_search('retrieval augmented generation', top_k=5)
print('Latency ms:', example['latency_ms'])
pd.DataFrame(example['results'])[['score','filename','page_start','citation','bm25_score','dense_score']].head()

## 6. FastAPI `/search` endpoint

The same endpoint is included in the submitted project package at `app/main.py`. This cell shows the API body you can test after running `uvicorn app.main:app --reload`.

In [ ]:
print('Run this in terminal:')
print('uvicorn app.main:app --reload')
print('
Then open http://localhost:8000/docs and test POST /search with:')
print(json.dumps({'query': 'What is graph retrieval?', 'top_k': 5, 'alpha': HYBRID_ALPHA}, indent=2))

## 7. Metrics table: Recall@k and p95 latency

In [ ]:
def load_questions(path):
    df = pd.read_excel(path, engine='openpyxl')
    df.columns = df.columns.str.strip().str.replace(' ', '_')
    if not {'Question', 'Correct_PDF'}.issubset(df.columns):
        raise ValueError('question.xlsx must contain Question and Correct_PDF columns')
    return df

questions_df = load_questions(QUESTIONS_PATH)
records = []
for _, row in tqdm(questions_df.iterrows(), total=len(questions_df), desc='Evaluating /search'):
    result = hybrid_search(row['Question'], top_k=5)
    returned = [r['filename'] for r in result['results']]
    rec = {
        'Question': row['Question'],
        'Correct_PDF': row['Correct_PDF'],
        'Top_Results': returned,
        'Top_Citations': [r['citation'] for r in result['results']],
        'Latency_ms': result['latency_ms'],
        'Recall@1': int(row['Correct_PDF'] in returned[:1]),
        'Recall@3': int(row['Correct_PDF'] in returned[:3]),
        'Recall@5': int(row['Correct_PDF'] in returned[:5]),
    }
    records.append(rec)

search_eval_df = pd.DataFrame(records)
metrics = {
    'Recall@1': search_eval_df['Recall@1'].mean(),
    'Recall@3': search_eval_df['Recall@3'].mean(),
    'Recall@5': search_eval_df['Recall@5'].mean(),
    'p95_latency_seconds': np.percentile(search_eval_df['Latency_ms'] / 1000.0, 95),
    'num_queries': len(search_eval_df),
}
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv('reports/d2_metrics_table.csv', index=False)
search_eval_df.to_csv('reports/d2_search_examples.csv', index=False)
print(metrics)
metrics_df

In [ ]:
# Top-k examples with citations for the report
examples = search_eval_df[['Question', 'Correct_PDF', 'Top_Results', 'Top_Citations', 'Latency_ms']].head(10)
examples

## 8. Neo4j graph build: Authors, Papers, Topics, Venues

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    session.run('CREATE CONSTRAINT paper_id IF NOT EXISTS FOR (p:Paper) REQUIRE p.paper_id IS UNIQUE')
    session.run('CREATE CONSTRAINT author_name IF NOT EXISTS FOR (a:Author) REQUIRE a.name IS UNIQUE')
    session.run('CREATE CONSTRAINT topic_name IF NOT EXISTS FOR (t:Topic) REQUIRE t.name IS UNIQUE')
    session.run('MATCH (n) DETACH DELETE n')
    for doc in documents:
        session.run('''
            MERGE (p:Paper {paper_id:$paper_id})
            SET p.title=$title, p.filename=$filename, p.year=$year, p.venue=$venue
        ''', **doc)
        for author in doc.get('authors', []) or ['Unknown Author']:
            session.run('''
                MERGE (a:Author {name:$author})
                WITH a MATCH (p:Paper {paper_id:$paper_id})
                MERGE (a)-[:WROTE]->(p)
            ''', author=author, paper_id=doc['paper_id'])
        for topic in doc.get('topics', []) or ['general ai']:
            session.run('''
                MERGE (t:Topic {name:$topic})
                WITH t MATCH (p:Paper {paper_id:$paper_id})
                MERGE (p)-[:ABOUT]->(t)
            ''', topic=topic, paper_id=doc['paper_id'])
        session.run('''
            MERGE (v:Venue {name:$venue})
            WITH v MATCH (p:Paper {paper_id:$paper_id})
            MERGE (p)-[:PUBLISHED_IN]->(v)
        ''', venue=doc.get('venue') or 'Unknown Venue', paper_id=doc['paper_id'])
print('Neo4j graph loaded.')

## 9. Five example Cypher queries

In [ ]:
example_cypher_queries = [
    ('Count nodes by label', 'MATCH (n) RETURN labels(n) AS labels, count(n) AS count ORDER BY count DESC', {}),
    ('List papers with topics', 'MATCH (p:Paper)-[:ABOUT]->(t:Topic) RETURN p.title AS paper, collect(t.name) AS topics LIMIT 10', {}),
    ('Find authors for paper title keyword', 'MATCH (a:Author)-[:WROTE]->(p:Paper) WHERE toLower(p.title) CONTAINS toLower($keyword) RETURN p.title AS paper, collect(a.name) AS authors LIMIT 10', {'keyword': 'learning'}),
    ('Find papers about a topic keyword', 'MATCH (p:Paper)-[:ABOUT]->(t:Topic) WHERE toLower(t.name) CONTAINS toLower($topic) RETURN t.name AS topic, p.title AS paper LIMIT 10', {'topic': 'retrieval'}),
    ('Find papers sharing the same topic', 'MATCH (p1:Paper)-[:ABOUT]->(t:Topic)<-[:ABOUT]-(p2:Paper) WHERE p1.paper_id <> p2.paper_id RETURN p1.title AS paper_a, t.name AS shared_topic, p2.title AS paper_b LIMIT 10', {}),
]

cypher_outputs = []
with driver.session() as session:
    for name, query, params in example_cypher_queries:
        rows = [r.data() for r in session.run(query, **params)]
        cypher_outputs.append({'name': name, 'cypher': query, 'rows': rows[:10]})
        print('
---', name, '---')
        print(query)
        print(rows[:3])

with open('reports/d2_cypher_examples.json', 'w', encoding='utf-8') as f:
    json.dump(cypher_outputs, f, indent=2)
print('Saved reports/d2_cypher_examples.json')

## 10. Dataflow diagram

Copy this Mermaid diagram into your report or GitHub README.

In [ ]:
diagram = '''flowchart LR
    A[PDF Corpus + question.xlsx] --> B[Ingestion: PDF text extraction]
    B --> C[Chunking with overlap + provenance]
    C --> D[(MongoDB: docs, chunks, provenance)]
    C --> E[Embedding model]
    E --> F[(Qdrant: dense vectors)]
    C --> G[BM25 lexical index]
    D --> H[Neo4j seed]
    H --> I[(Neo4j: Authors Papers Topics Venues)]
    J[/FastAPI /search/] --> G
    J --> F
    G --> K[Hybrid score fusion]
    F --> K
    I --> L[Cypher graph queries]
    K --> M[Top-k chunks + citations]
    L --> M
'''
Path('reports/d2_dataflow_diagram.mmd').write_text(diagram, encoding='utf-8')
print(diagram)

```mermaid
flowchart LR
    A[PDF Corpus + question.xlsx] --> B[Ingestion: PDF text extraction]
    B --> C[Chunking with overlap + provenance]
    C --> D[(MongoDB: docs, chunks, provenance)]
    C --> E[Embedding model]
    E --> F[(Qdrant: dense vectors)]
    C --> G[BM25 lexical index]
    D --> H[Neo4j seed]
    H --> I[(Neo4j: Authors Papers Topics Venues)]
    J[/FastAPI /search/] --> G
    J --> F
    G --> K[Hybrid score fusion]
    F --> K
    I --> L[Cypher graph queries]
    K --> M[Top-k chunks + citations]
    L --> M
```

## 11. D2 submission checklist

- Upload the project package files to GitHub.
- Include `docker-compose.yml`, `Dockerfile`, `.env.example`, `requirements.txt`, `app/`, `src/`, and `scripts/`.
- Run `python scripts/seed_stores.py`, `python scripts/seed_neo4j.py`, and `python scripts/evaluate_search.py`.
- Include the generated metrics and search examples from `reports/`.
- Include this notebook as evidence that all D2 requirements were implemented.